# 1.0 — Recebimento e movimentação de arquivos

- **Propósito:** Direcionar arquivos da área de entrada para os volumes correspondentes conforme o prefixo do nome.
- **Entrada:** Volume de entrada do orquestrador
- **Saídas:** Volumes definidos em `MAPPING`
- **Carga:** Sob demanda · **Modo:** Simulação ou execução

In [0]:
# Módulos necessários para manipulação de paths e sistema de arquivos
import os
from pathlib import Path

In [0]:
# Widgets de configuração
# source_volume: caminho do volume de origem
# modo_execucao: testar (simulação) ou executar (move arquivos)
dbutils.widgets.text("source_volume", "/Volumes/parts_hdbk_sandbox/_file_orchestrator/demand/01_apuracao_demanda/in/", "1 Volume de Origem")
dbutils.widgets.dropdown("modo_execucao", "testar", ["testar", "executar"], "2 Modo de Execução")

source_volume = dbutils.widgets.get("source_volume")
modo_execucao = dbutils.widgets.get("modo_execucao")

# Configuração carregada dos widgets

In [0]:
# Mapeamento de prefixos para volumes de destino
# Adicione novos mapeamentos no formato: 'prefixo': '/Volumes/catalog/schema/volume'

MAPPING = {
    'kna1': '/Volumes/parts_hdbk_sandbox/dm_customers/kna1_sap',
    'KNA1': '/Volumes/parts_hdbk_sandbox/dm_customers/kna1_sap',
    # Adicione mais mapeamentos abaixo:
    # 'vbak': '/Volumes/parts_hdbk_sandbox/dm_sales/vbak_sap',
    # 'mara': '/Volumes/parts_hdbk_sandbox/dm_materials/mara_sap',
}

# Mapeamentos configurados

In [0]:
# Funções para determinar destino e mover arquivos baseado em prefixo
def get_destination_path(prefix):
    """
    Retorna o caminho de destino baseado no prefixo do arquivo.
    
    Args:
        prefix: Prefixo extraído do nome do arquivo (texto antes do primeiro '_')
    
    Returns:
        Caminho completo do volume de destino
        
    Raises:
        ValueError: Se o prefixo não estiver no mapeamento
    """
    destination = MAPPING.get(prefix)
    
    if destination is None:
        raise ValueError(f"Prefixo '{prefix}' não encontrado no mapeamento. Adicione-o à constante MAPPING.")
    
    return destination


def move_files_by_prefix(source_volume, dry_run=True):
    """
    Move arquivos do volume origem para volumes destino baseado no prefixo do nome.
    
    Args:
        source_volume: Caminho completo do volume origem
        dry_run: Se True, apenas simula sem mover. Se False, move de fato.
    
    Returns:
        Dict com estatísticas da operação
    """
    stats = {
        'total_files': 0,
        'moved': 0,
        'errors': [],
        'details': []
    }
    
    # Lista todos os arquivos no volume origem
    try:
        files = dbutils.fs.ls(source_volume)
    except Exception as e:
        stats['errors'].append(f"Erro ao listar arquivos em {source_volume}: {str(e)}")
        return stats
    
    # Filtra apenas arquivos (não diretórios)
    files = [f for f in files if not f.isDir()]
    stats['total_files'] = len(files)
    
    modo = "SIMULAÇÃO" if dry_run else "EXECUÇÃO REAL"
    print(f"Encontrados {len(files)} arquivo(s) em {source_volume}")
    print(f"Modo: {modo}\n")
    
    for file_info in files:
        file_path = file_info.path
        file_name = os.path.basename(file_path)
        
        # Extrai o prefixo:
        # - Se houver '_': usa tudo antes do primeiro '_'
        # - Se não houver '_': usa o nome do arquivo sem extensão
        if '_' in file_name:
            prefix = file_name.split('_')[0]
        else:
            # Remove extensão do arquivo para usar como prefixo
            prefix = os.path.splitext(file_name)[0]
        
        try:
            # Obtém o caminho de destino
            destination_volume = get_destination_path(prefix)
            destination_file = f"{destination_volume}/{file_name}"
            
            # Mostra o que será feito
            print(f"{'[TESTE]' if dry_run else '[MOVENDO]'}: {file_name}")
            print(f"  De: {file_path}")
            print(f"  Para: {destination_file}")
            
            if not dry_run:
                # Move o arquivo (copia + deleta origem)
                # Sobrescreve automaticamente se já existir arquivo com mesmo nome
                dbutils.fs.cp(file_path, destination_file, recurse=False)
                dbutils.fs.rm(file_path)
                print(f"  ✓ Movido com sucesso\n")
            else:
                print(f"  ✓ Seria movido (modo teste)\n")
            
            stats['moved'] += 1
            stats['details'].append({
                'file': file_name,
                'prefix': prefix,
                'destination': destination_volume,
                'status': 'success' if not dry_run else 'simulated'
            })
            
        except ValueError as e:
            # Prefixo não mapeado
            error_msg = f"ERRO: {file_name} - {str(e)}"
            print(error_msg + "\n")
            stats['errors'].append(error_msg)
            
        except Exception as e:
            # Erro genérico
            error_msg = f"ERRO: {file_name} - {str(e)}"
            print(error_msg + "\n")
            stats['errors'].append(error_msg)
    
    return stats


def print_relatorio(results, modo):
    """Imprime relatório final da operação."""
    print("="*60)
    print(f"RELATÓRIO FINAL - {modo.upper()}")
    print("="*60)
    print(f"Total de arquivos encontrados: {results['total_files']}")
    print(f"Arquivos processados: {results['moved']}")
    print(f"Erros/Skips: {len(results['errors'])}")
    
    if results['errors']:
        print("\nDetalhes dos erros:")
        for error in results['errors']:
            print(f"  - {error}")


print("✓ Funções auxiliares carregadas")

In [0]:
# Executa a movimentação de arquivos baseada no modo configurado


# Define se é teste ou execução real baseado no widget
dry_run = (modo_execucao == "executar")

# Executa a movimentação
results = move_files_by_prefix(source_volume, dry_run=dry_run)

# Relatório final
print_relatorio(results, modo_execucao)